# Inverse Diagnostic Demo\n\nЭтот notebook использует только публичный API `run_inverse_diagnostic_cycle(...)` и notebook-loader без дублирования бизнес-логики.

In [ ]:
from pathlib import Path\nimport pandas as pd\n\nfrom hidden_patterns_combat.app.inverse_diagnostic_cycle import run_inverse_diagnostic_cycle\nfrom hidden_patterns_combat.ui.inverse_notebook import (\n    display_inverse_plots,\n    display_inverse_report,\n    load_inverse_artifacts,\n)

In [ ]:
repo_root = Path.cwd()\ninput_path = repo_root / "data" / "raw" / "episodes.xlsx"\nbase_output_dir = repo_root / "artifacts" / "inverse_diagnostic_runs"\n\nisolated_run = True\ncleanup_mode = "none"\n

In [ ]:
result = run_inverse_diagnostic_cycle(\n    input_path=input_path,\n    output_dir=base_output_dir,\n    isolated_run=isolated_run,\n    cleanup_mode=cleanup_mode,\n    retrain=True,\n    n_states=3,\n    topology_mode="left_to_right",\n    generate_plots=True,\n    verbose=True,\n)\n\nfinal_output_dir = Path(result.final_output_dir)\nartifacts = load_inverse_artifacts(\n    final_output_dir,\n    expected_run_id=result.run_id,\n    expected_run_fingerprint=result.run_fingerprint,\n)

## Current Run Metadata

In [ ]:
try:\n    from IPython.display import Markdown, display\n\n    manifest = artifacts.run_manifest or {}\n    warning_lines = "\n".join(f"- {w}" for w in artifacts.loader_warnings[:20])\n    if not warning_lines:\n        warning_lines = "- none"\n\n    display(Markdown(\n        f"""\n- run_id: `{result.run_id}`\n- final_output_dir: `{result.final_output_dir}`\n- run_manifest_path: `{result.run_manifest_path}`\n- started_at: `{manifest.get('started_at', 'n/a')}`\n- finished_at: `{manifest.get('finished_at', 'n/a')}`\n- input_file: `{manifest.get('input_file_name', 'n/a')}`\n- key warnings:\n{warning_lines}\n"""\n    ))\nexcept Exception:\n    print("run_id:", result.run_id)\n    print("final_output_dir:", result.final_output_dir)\n    print("run_manifest_path:", result.run_manifest_path)\n    print("started_at:", artifacts.run_manifest.get("started_at", "n/a"))\n    print("finished_at:", artifacts.run_manifest.get("finished_at", "n/a"))\n    print("input_file:", artifacts.run_manifest.get("input_file_name", "n/a"))\n    print("warnings:")\n    for warning in artifacts.loader_warnings[:20]:\n        print("-", warning)

In [ ]:
artifacts.artifact_status

In [ ]:
display_inverse_report(artifacts.report_markdown)

In [ ]:
display_inverse_plots(artifacts.plot_paths)

In [ ]:
if not artifacts.episode_analysis.empty:\n    preview_cols = [\n        c\n        for c in [\n            "episode_id",\n            "sequence_id",\n            "observed_zap_class",\n            "hidden_state_name",\n            "confidence",\n        ]\n        if c in artifacts.episode_analysis.columns\n    ]\n    display(artifacts.episode_analysis[preview_cols].head(20))\nelse:\n    print("episode_analysis.csv is missing or empty")